## Sub-graphs

### Goal

- [More about Subgraphs](https://docs.langchain.com/oss/python/langgraph/use-subgraphs)

## State
- Sub-graphs allow you to create and manage different states in different parts of your graph.
- This is particularly useful for multi-agent systems, with teams of agents that each have their own state.
- Let's consider a toy example:
    - I have a system that accepts logs
    - It performs two separate sub-tasks by different agents (summarize logs, find failure modes)
    - I want to perform these two operations in two different sub-graphs.

The most critical thing to understand is how these graphs communicate

- In short, communication is done with over-lapping keys:
  - The Sub-graph can access docs from the parent
  - The parent can access summary/failure_report from sub-graphs



In [1]:
from operator import add
from typing_extensions import TypedDict
from typing import List, Optional, Annotated

class Log(TypedDict):
    id: str
    question: str
    docs: Optional[List]
    answer: str
    grade: Optional[int]
    grader: Optional[int]
    feedback: Optional[str]



In [2]:
log = Log(
    id="1", question="What is your name", docs = ["docs1", "docs2"],
    answer="My name is bilal"
)

In [3]:
log

{'id': '1',
 'question': 'What is your name',
 'docs': ['docs1', 'docs2'],
 'answer': 'My name is bilal'}

## Subgraphs
Here is the failure analysis sub-graph, which uses FailureAnalysisState.


In [4]:
from IPython.display import Image, display 
from langgraph.graph import StateGraph, START, END


# Failure Analysis Sub-graph

class FailureAnalysisState(TypedDict):
    cleaned_logs: List[Log]
    failure_report: List[Log]
    fa_summary: str
    processed_logs: List[str]

class FailureAnalysisOutputState(TypedDict):
    fa_summary: str
    processed_logs: List[str]

In [5]:
failure_analysis_state: FailureAnalysisState = {
    # INPUT — received from the parent graph (overlapping key)
    "cleaned_logs": [
        Log(id="1", question="Why is the API slow?",  docs=["doc1"], answer="Timeout issue",  grade=0, grader=1, feedback="Wrong"),
        Log(id="2", question="What is the DB error?", docs=["doc2"], answer="Connection lost", grade=0, grader=1, feedback="Incorrect"),
        Log(id="3", question="What is caching?",      docs=["doc3"], answer="Caching stores data temporarily", grade=1, grader=1, feedback="Correct"),
    ],

    # INTERMEDIATE — computed inside the sub-graph
    "failure_report": [
        Log(id="1", question="Why is the API slow?",  docs=["doc1"], answer="Timeout issue",  grade=0, grader=1, feedback="Wrong"),
        Log(id="2", question="What is the DB error?", docs=["doc2"], answer="Connection lost", grade=0, grader=1, feedback="Incorrect"),
    ],

    # OUTPUTS — will be passed back to the parent graph
    "fa_summary": "2 out of 3 logs show failures: API timeouts and DB connection errors.",
    "processed_logs": ["log-1", "log-2", "log-3"],
}

In [6]:
failure_analysis_state

{'cleaned_logs': [{'id': '1',
   'question': 'Why is the API slow?',
   'docs': ['doc1'],
   'answer': 'Timeout issue',
   'grade': 0,
   'grader': 1,
   'feedback': 'Wrong'},
  {'id': '2',
   'question': 'What is the DB error?',
   'docs': ['doc2'],
   'answer': 'Connection lost',
   'grade': 0,
   'grader': 1,
   'feedback': 'Incorrect'},
  {'id': '3',
   'question': 'What is caching?',
   'docs': ['doc3'],
   'answer': 'Caching stores data temporarily',
   'grade': 1,
   'grader': 1,
   'feedback': 'Correct'}],
 'failure_report': [{'id': '1',
   'question': 'Why is the API slow?',
   'docs': ['doc1'],
   'answer': 'Timeout issue',
   'grade': 0,
   'grader': 1,
   'feedback': 'Wrong'},
  {'id': '2',
   'question': 'What is the DB error?',
   'docs': ['doc2'],
   'answer': 'Connection lost',
   'grade': 0,
   'grader': 1,
   'feedback': 'Incorrect'}],
 'fa_summary': '2 out of 3 logs show failures: API timeouts and DB connection errors.',
 'processed_logs': ['log-1', 'log-2', 'log-3']